# Class 9 Offline Content Generator for Kaggle

This notebook extracts class 9 textbook chapters sequentially, prefers Docling for PDF extraction, and falls back to PyMuPDF if Docling is unavailable. It then generates:

- `summary.json`
- `concepts.json`
- `flashcards.json`
- `quizzes.json`

The prompts are tuned for educational quality and require at least 20 MCQs per chapter.

In [ ]:
%pip install -q transformers accelerate bitsandbytes sentencepiece pymupdf
%pip install -q docling || true

In [ ]:
import json
import os
import re
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from docling.document_converter import DocumentConverter
    DOCLING_AVAILABLE = True
except Exception:
    DOCLING_AVAILABLE = False

TEXTBOOKS_ROOT = Path(os.environ.get('TEXTBOOKS_ROOT', '/kaggle/input/textbooks'))
if not TEXTBOOKS_ROOT.exists():
    input_root = Path('/kaggle/input')
    candidates = [p for p in input_root.iterdir() if p.is_dir() and 'textbook' in p.name.lower()]
    if candidates:
        TEXTBOOKS_ROOT = candidates[0]

OUTPUT_ROOT = Path('/kaggle/working/class9_generation_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_CANDIDATES = [
    os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct'),
    'Qwen/Qwen2.5-3B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
]
MAX_CHARS = int(os.environ.get('MAX_CHARS', '14000'))
FLASHCARD_COUNT = 12
QUIZ_COUNT = 20
CONCEPT_COUNT = 8
MAX_CHAPTERS = None  # set to 1 for a smoke test
ONLY_SLUGS = None    # set to a list like ['number_systems'] to run selected chapters only


In [ ]:
def dump_model(value):
    if hasattr(value, 'export_to_dict'):
        return value.export_to_dict()
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if isinstance(value, dict):
        return value
    return {}


def normalize_space(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '').replace(' ', ' ')).strip()


def pretty_title(stem: str) -> str:
    text = stem.replace('_', ' ').replace('-', ' ')
    text = re.sub(r'_s', "'s", text)
    text = normalize_space(text)
    return text.title().replace("'S", "'s")


def looks_like_class9_pdf(path: Path) -> bool:
    parts = [p.lower() for p in path.parts]
    joined = ' / '.join(parts)
    return any(token in joined for token in ['class 9', 'class_9', '9th social science'])


def discover_pdfs(root: Path):
    pdfs = sorted([p for p in root.rglob('*.pdf') if p.is_file() and looks_like_class9_pdf(p)], key=lambda p: str(p).lower())
    return pdfs


def source_slug(pdf_path: Path) -> str:
    return pdf_path.stem.lower().replace(' ', '_')


In [ ]:
def extract_with_docling(pdf_path: Path) -> tuple[str, str]:
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    document = dump_model(result.document)
    parts = []
    title = document.get('name') or pdf_path.stem

    texts = document.get('texts') if isinstance(document.get('texts'), list) else []
    for item in texts:
        if not isinstance(item, dict):
            continue
        text = item.get('text') or item.get('orig') or item.get('content')
        if isinstance(text, str) and text.strip():
            parts.append(normalize_space(text))

    tables = document.get('tables') if isinstance(document.get('tables'), list) else []
    for idx, table in enumerate(tables, start=1):
        if not isinstance(table, dict):
            continue
        data = table.get('data') if isinstance(table.get('data'), dict) else table
        cells = data.get('table_cells') or data.get('cells') or []
        rows = {}
        for cell in cells if isinstance(cells, list) else []:
            if not isinstance(cell, dict):
                continue
            row = int(cell.get('start_row_offset_idx') or cell.get('row') or 0)
            col = int(cell.get('start_col_offset_idx') or cell.get('col') or 0)
            txt = normalize_space(cell.get('text') or '')
            rows.setdefault(row, []).append((col, txt))
        if rows:
            rendered = [' | '.join(t for _, t in sorted(cols)) for _, cols in sorted(rows.items())]
            parts.append(f'Table {idx}:\n' + '\n'.join(rendered))

    pictures = document.get('pictures') if isinstance(document.get('pictures'), list) else []
    for idx, picture in enumerate(pictures, start=1):
        if not isinstance(picture, dict):
            continue
        caption = normalize_space(picture.get('caption') or picture.get('text') or '')
        if caption:
            parts.append(f'Figure {idx}: {caption}')

    text = '\n\n'.join([p for p in parts if p])
    return pretty_title(Path(pdf_path).stem), text


def extract_with_pymupdf(pdf_path: Path) -> tuple[str, str]:
    import fitz
    parts = []
    with fitz.open(pdf_path) as pdf:
        for page in pdf:
            text = page.get_text('text')
            text = normalize_space(text)
            if text:
                parts.append(text)
    return pretty_title(Path(pdf_path).stem), '\n\n'.join(parts)


def extract_chapter_source(pdf_path: Path) -> tuple[str, str]:
    if DOCLING_AVAILABLE:
        try:
            title, text = extract_with_docling(pdf_path)
            if text.strip():
                return title, text
        except Exception:
            pass
    return extract_with_pymupdf(pdf_path)


def trim_source(text: str, max_chars: int = MAX_CHARS) -> str:
    text = normalize_space(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars]


In [ ]:
def load_model():
    last_error = None
    for model_id in MODEL_CANDIDATES:
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            use_4bit = torch.cuda.is_available()
            quant_config = None
            dtype = torch.float16 if torch.cuda.is_available() else torch.float32
            if use_4bit:
                quant_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type='nf4',
                    bnb_4bit_compute_dtype=dtype,
                )

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                device_map='auto' if torch.cuda.is_available() else None,
                torch_dtype=dtype,
                quantization_config=quant_config,
                trust_remote_code=True,
            )
            return model_id, tokenizer, model
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'Could not load any model candidate. Last error: {last_error}')


MODEL_ID, tokenizer, model = load_model()
print(f'Loaded: {MODEL_ID}')


In [ ]:
SYSTEM_PROMPT = '''You are an expert curriculum writer for grades 6-10.
Write only factual, educational content grounded in the provided textbook text.
Do not invent facts.
Keep language age-appropriate and exam-ready.
Return valid JSON only, with no markdown fences, no commentary, and no trailing text.'''


def extract_json(text: str):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('
', 1)[1] if '
' in text else text
        if text.endswith('```'):
            text = text[:-3]
    first = text.find('{')
    last = text.rfind('}')
    if first != -1 and last != -1 and last > first:
        text = text[first:last+1]
    return json.loads(text)


def generate_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 3000, temperature: float = 0.2, retries: int = 2):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    last_error = None
    for attempt in range(retries + 1):
        try:
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=temperature > 0,
                    temperature=temperature,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
            text = tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            return extract_json(text)
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                user_prompt = user_prompt + f'\n\nYour previous answer was invalid because: {exc}. Return corrected JSON only.'
                prompt = tokenizer.apply_chat_template([
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt},
                ], tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(prompt, return_tensors='pt')
                if torch.cuda.is_available():
                    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    raise last_error


In [ ]:
def prompt_summary(title: str, source: str) -> str:
    return f'''Create a JSON object with these keys:
- chapter_title: string
- overview: 180-260 words, clear and grounded in the text
- key_points: exactly 5 concise bullet-style strings

Rules:
- Focus on the actual chapter content.
- Mention core concepts, formulas, or civic ideas when present.
- Do not add facts that are not in the source.

Chapter title: {title}

Source text:
{source}
'''


def prompt_concepts(title: str, source: str) -> str:
    return f'''Create a JSON array of exactly {CONCEPT_COUNT} items.
Each item must have:
- concept: short concept name
- description: 2-3 sentences explaining it in student-friendly language

Rules:
- Pick the most teachable concepts from the chapter.
- Prefer core terms, laws, definitions, theorems, processes, institutions, or formulas.
- Avoid trivial or duplicate concepts.
- Stay faithful to the source.

Chapter title: {title}

Source text:
{source}
'''


def prompt_flashcards(title: str, source: str) -> str:
    return f'''Create a JSON array of exactly {FLASHCARD_COUNT} flashcards.
Each item must have:
- front: a useful study question
- back: a concise but complete answer

Rules:
- Use a mix of definitions, why/how questions, comparisons, formulas, and applications.
- Make the flashcards useful for grades 6-10 students.
- Do not make the questions trivial.
- Ground every answer in the source.

Chapter title: {title}

Source text:
{source}
'''


def prompt_quizzes(title: str, source: str) -> str:
    return f'''Create a JSON array of exactly {QUIZ_COUNT} MCQs.
Each item must have:
- question: a clear question
- options: exactly 4 distinct answer options
- answer: the correct option text exactly as it appears in options
- explanation: 1-3 sentences explaining why the answer is correct

Rules:
- Questions should be educationally useful, not trivia.
- Mix recall, understanding, and simple application.
- Distractors must be plausible and aligned with the chapter.
- At least some questions should test formulas, definitions, causes, properties, or real-world application.
- Do not repeat the same question pattern too often.
- Stay faithful to the source.

Chapter title: {title}

Source text:
{source}
'''


def validate_mcqs(items):
    if not isinstance(items, list) or len(items) != QUIZ_COUNT:
        return False
    for item in items:
        if not isinstance(item, dict):
            return False
        if not all(key in item for key in ['question', 'options', 'answer', 'explanation']):
            return False
        if not isinstance(item['options'], list) or len(item['options']) != 4:
            return False
        if item['answer'] not in item['options']:
            return False
    return True


def validate_flashcards(items):
    return isinstance(items, list) and len(items) == FLASHCARD_COUNT and all(isinstance(x, dict) and 'front' in x and 'back' in x for x in items)


def validate_concepts(items):
    return isinstance(items, list) and len(items) == CONCEPT_COUNT and all(isinstance(x, dict) and 'concept' in x and 'description' in x for x in items)


def validate_summary(payload):
    return isinstance(payload, dict) and 'chapter_title' in payload and 'overview' in payload and 'key_points' in payload and isinstance(payload['key_points'], list) and len(payload['key_points']) == 5


In [ ]:
def process_chapter(pdf_path: Path):
    slug = source_slug(pdf_path)
    chapter_dir = OUTPUT_ROOT / slug
    chapter_dir.mkdir(parents=True, exist_ok=True)

    existing = [chapter_dir / name for name in ['summary.json', 'concepts.json', 'flashcards.json', 'quizzes.json']]
    if all(path.exists() for path in existing):
        return {'pdf': str(pdf_path), 'slug': slug, 'status': 'skipped'}

    title, source_text = extract_chapter_source(pdf_path)
    source_text = trim_source(source_text)

    summary = generate_json(SYSTEM_PROMPT, prompt_summary(title, source_text), max_new_tokens=1200, temperature=0.15, retries=2)
    if not validate_summary(summary):
        raise ValueError(f'Invalid summary for {slug}')

    concepts = generate_json(SYSTEM_PROMPT, prompt_concepts(title, source_text), max_new_tokens=1800, temperature=0.2, retries=2)
    if not validate_concepts(concepts):
        raise ValueError(f'Invalid concepts for {slug}')

    flashcards = generate_json(SYSTEM_PROMPT, prompt_flashcards(title, source_text), max_new_tokens=2600, temperature=0.25, retries=2)
    if not validate_flashcards(flashcards):
        raise ValueError(f'Invalid flashcards for {slug}')

    quizzes = generate_json(SYSTEM_PROMPT, prompt_quizzes(title, source_text), max_new_tokens=5000, temperature=0.2, retries=2)
    if not validate_mcqs(quizzes):
        raise ValueError(f'Invalid quizzes for {slug}')

    (chapter_dir / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    (chapter_dir / 'concepts.json').write_text(json.dumps(concepts, indent=2, ensure_ascii=False), encoding='utf-8')
    (chapter_dir / 'flashcards.json').write_text(json.dumps(flashcards, indent=2, ensure_ascii=False), encoding='utf-8')
    (chapter_dir / 'quizzes.json').write_text(json.dumps(quizzes, indent=2, ensure_ascii=False), encoding='utf-8')

    return {'pdf': str(pdf_path), 'slug': slug, 'title': title, 'status': 'generated'}


In [ ]:
pdfs = discover_pdfs(TEXTBOOKS_ROOT)
if ONLY_SLUGS:
    wanted = {s.lower() for s in ONLY_SLUGS}
    pdfs = [p for p in pdfs if source_slug(p) in wanted]

print(f'Found {len(pdfs)} class 9 pdf(s) under {TEXTBOOKS_ROOT}')
results = []
for index, pdf_path in enumerate(pdfs, start=1):
    if MAX_CHAPTERS is not None and index > MAX_CHAPTERS:
        break
    print(f'[{index}/{len(pdfs)}] {pdf_path.name}')
    try:
        results.append(process_chapter(pdf_path))
    except Exception as exc:
        results.append({'pdf': str(pdf_path), 'slug': source_slug(pdf_path), 'status': 'failed', 'error': str(exc)})
        print(f'  FAILED: {exc}')

summary_counts = {}
for row in results:
    summary_counts[row['status']] = summary_counts.get(row['status'], 0) + 1
print(summary_counts)


In [ ]:
import shutil
zip_path = Path('/kaggle/working/class9_generation_outputs.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive('/kaggle/working/class9_generation_outputs', 'zip', OUTPUT_ROOT)
print(f'Wrote {zip_path}')


## Notes

- Set `MAX_CHAPTERS = 1` for a smoke test.
- Set `ONLY_SLUGS = ['number_systems']` to run a single chapter.
- The notebook prefers Docling extraction and falls back to PyMuPDF if needed.
- If `Qwen/Qwen2.5-7B-Instruct` does not fit, the notebook falls back to smaller instruct models automatically.